### Analyze Visium fluorescence data

This tutorial shows how to apply Squidpy image analysis features for the analysis of Visium data.

For a tutorial using Visium data that includes the graph analysis functions, have a look at Analyze Visium H&E data. The dataset used here consists of a Visium slide of a coronal section of the mouse brain. The original dataset is publicly available at the 10x genomics dataset portal <https://support.10xgenomics.com/spatial-gene-expression/datasets>_ . Here, we provide a pre-processed dataset, with pre-annotated clusters, in anndata.AnnData format and the tissue image in squidpy.im.ImageContainer format.

<div class="alert alert-block alert-success">
    <b>✅ A couple of notes on pre-processing:</b>
</div>

- The pre-processing pipeline is the same as the one shown in the original [Scanpy tutorial](https://scanpy-tutorials.readthedocs.io/en/latest/spatial/basic-analysis.html)
- The cluster annotation was performed using several resources, such as 
  - the [Allen Brain Atlas](https://mouse.brain-map.org/experiment/thumbnails/100048576?image_type=atlas) 
  - the [Mouse Brain gene expression atlas](http://mousebrain.org/) from the Linnarson lab.


In [ ]:
import pandas as pd

import matplotlib.pyplot as plt

import anndata as ad
import scanpy as sc
import squidpy as sq

In [ ]:
sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

### load the pre-processed dataset

In [ ]:
img = sq.datasets.visium_fluo_image_crop()
adata = sq.datasets.visium_fluo_adata_crop()

### Visualize the cluster annotation in the spatial context with squidpy.pl.spatial_scatter().

As you can see, this dataset is a smaller crop of the whole brain section. We provide this crop to make the execution time of this tutorial a bit shorter.

In [ ]:
plt.rcParams['figure.figsize'] = (10,6)
# plt.rcParams['figure.dpi'] = 240

sq.pl.spatial_scatter(adata, color="cluster")


### The fluorescence image provided with this dataset has three channels

1. DAPI (specific to DNA)
2. anti-NEUN (specific to neurons)
3. anti-GFAP (specific to Glial cells). 

We can directly visualize the channels with the method squidpy.im.ImageContainer.show().

In [ ]:
img.show(channelwise=True, figsize=(14,10))

In [ ]:
img.__dict__

### Are there only 3 images? how to print each separately?

Not 3 images — it's one multi-channel image held in a single ImageContainer. The fluorescence image is a stack of marker channels (DAPI + antibody stains), and each "image" you might be seeing is actually one channel of that stack, not a separate layer.

First, inspect what's really there:

In [ ]:
print(img)                      # shows layers + shape, e.g. ImageContainer[shape=(7272, 7272), layers=['image']]

In [ ]:
print(img["image"].dims)        # ('y', 'x', 'z', 'channels')
print(img["image"].shape)       # last dim = number of channels
n_ch = img["image"].sizes["channels"]
print("channels:", n_ch)        # this fluo crop has 3

### One specific channel (antibody)

In [ ]:
img.show(layer="image", channel=0, figsize=(8,8))   # DAPI

In [ ]:
arr = img["image"]                       # dims (y, x, z, channels)
n_ch = img["image"].sizes["channels"]

fig, axes = plt.subplots(1, n_ch, figsize=(5 * n_ch, 5))

for c in range(n_ch):
    axes[c].imshow(arr.isel(z=0, channels=c), cmap="gray")
    axes[c].set_title(f"channel {c}")
    axes[c].axis("off")

plt.tight_layout()
plt.show()

### Expression

Given the DAPI image
Given the most highly expressed genes related to mitochondria
Get the first gene expression
Plot the overlap of the image cluster + expression dots 

In [ ]:
print(adata)                    # overview: X, obs, var, obsm, uns
print(adata.shape)              # (704, 16562)  -> spots x genes

In [ ]:
adata.X[:5, :5].toarray() if hasattr(adata.X, "toarray") else adata.X[:5, :5]

In [ ]:
X = adata.X
mx = X.max()
print("max value:", mx, "| looks like:", "counts" if float(mx).is_integer() else "normalized/log")
print("raw present:", adata.raw is not None)   # counts often stashed in adata.raw
print("layers:", list(adata.layers.keys()))

In [ ]:
adata.obs.columns          # includes 'cluster' -> the anatomical annotations

In [ ]:
adata.obsm['spatial']      # spot xy coordinates (align to the image)

In [ ]:
adata.uns['spatial']       # library id, image scalefactors, hires/lowres thumbnails

In [ ]:
dfc = adata.var           # gene metadata
dfc.columns

In [ ]:
dfc.head(3).T

### Mitochondrial gene expression

In [ ]:
# 1. DAPI = channel 0 of the fluorescence stack (2D array, y-by-x)
dapi = img["image"].isel(z=0, channels=0).values

In [ ]:
# 2. Mitochondrial genes, ranked by mean expression across spots
is_mito = adata.var_names.str.lower().str.startswith("mt-")
mito = adata.var_names[is_mito]
print("mito genes:", list(mito))

In [ ]:
Xm = adata[:, mito].X
Xm = Xm.toarray() if hasattr(Xm, "toarray") else np.asarray(Xm)
mean_expr = pd.Series(Xm.mean(axis=0), index=mito).sort_values(ascending=False)
print(mean_expr)

In [ ]:
# 3. First (most highly expressed) mito gene -> its per-spot vector
top_gene = mean_expr.index[0]
expr = adata[:, top_gene].X
expr = expr.toarray().ravel() if hasattr(expr, "toarray") else np.asarray(expr).ravel()

### Overlay: DAPI image + expression dots at spot coordinates

In [ ]:
xy = adata.obsm["spatial"]              # pixel coords aligned to the full-res crop
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(dapi, cmap="gray")
pts = ax.scatter(xy[:, 0], xy[:, 1], c=expr, cmap="autumn_r", s=18, linewidths=0)
# pts = ax.scatter(xy[:, 0], xy[:, 1], c=expr, cmap="YlOrRd", s=18, linewidths=0)
# pts = ax.scatter(xy[:, 0], xy[:, 1], c=expr, cmap="viridis", s=18, linewidths=0)
plt.colorbar(pts, ax=ax, shrink=0.7, label=f"{top_gene}")
ax.set_title(f"DAPI + {top_gene} expression")
ax.axis("off")
plt.tight_layout()
plt.show()